In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

class SelectiveStandardScaler(BaseEstimator, TransformerMixin):
    """
    Scale only selected columns; pass all others through unchanged.
    """
    def __init__(self, scale_cols):
        self.scale_cols = scale_cols
        self.scaler = StandardScaler()

    def fit(self, X, y=None):
        self.scaler.fit(X[self.scale_cols])
        return self

    def transform(self, X):
        X_out = X.copy()
        X_out[self.scale_cols] = self.scaler.transform(X[self.scale_cols])
        return X_out

def agg_observations(df):
    grouped = df.groupby("obs")

    numeric_cols = df.select_dtypes(include="number").columns
    agg_numeric = grouped[numeric_cols].mean()

    meta_cols = ["time", "sub_id", "cat_0", "cat_1", "cat_2", "cat_3", "cat_4"]
    agg_meta = grouped[meta_cols].first()

    return pd.concat([agg_numeric, agg_meta], axis=1)

def attach_time_and_sort(df):
    df = df.copy()

    df.sort_values("time", inplace=True)

    int_cols = ["sub_id", "cat_0", "cat_1", "cat_2", "cat_3", "cat_4"]
    df[int_cols] = df[int_cols].astype(int)

    return df

post_process = FunctionTransformer(attach_time_and_sort, validate=False)


pipeline = Pipeline(steps=[
    ("to_numeric", FunctionTransformer(
        lambda df: df.apply(pd.to_numeric, errors="ignore"),
        validate=False
    )),
    ("aggregate_observations", FunctionTransformer(
        agg_observations,
        validate=False
    )),
    ("post_process", FunctionTransformer(
        attach_time_and_sort,
        validate=False
    )),
    ("scale", StandardScaler())
])

df = pd.read_csv("Project_Data_export/train_competition_2026.csv")
df.sort_values("time", inplace=True)

# Extract datetime only
time_col = df["time"]

processed_data = pipeline.fit_transform(df)

# Reattach time aligned by obs
time_mapping = time_col.groupby(df["obs"]).first()
processed_data["time"] = processed_data.index.map(time_mapping)
processed_data.sort_values("time", inplace=True)
processed_data[['sub_id', 'cat_0', 'cat_1', 'cat_2', 'cat_3', 'cat_4']] = processed_data[['sub_id', 'cat_0', 'cat_1', 'cat_2', 'cat_3', 'cat_4']].astype(int)
processed_data.head()


,sub_id,time,num_0,num_1,num_2,cat_0,cat_1,cat_2,cat_3,cat_4,t_0,t_1,t_2,t_3,t_4,y_1,y_2
obs,,,,,,,,,,,,,,,,,
9953,1021,2043-08-12 13:33:22,-0.093583,-0.211482,-0.151129,0,2,0,0,1,-0.265702,0.232294,-0.425251,-0.702941,-0.448514,37.64,77.40
9954,1021,2043-08-12 18:20:22,-0.093583,-0.211482,-0.151129,0,2,0,0,1,-0.192377,0.115235,-0.697081,-1.515308,-1.090151,27.98,76.82
9955,1021,2043-08-12 20:22:22,-0.093583,-0.211482,-0.151129,0,2,0,0,1,-0.233228,0.125404,-0.084412,-0.996599,-0.460502,41.73,82.05
1513,132,2043-09-03 06:33:53,-1.968018,0.153046,0.679177,0,6,0,0,0,3.027694,0.327377,-0.208972,0.664390,0.211459,46.74,125.04
1514,132,2043-09-03 09:43:53,-1.968018,0.153046,0.679177,0,6,0,0,0,2.299069,0.140789,-0.422735,0.906359,0.283286,49.56,116.26
